In [22]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================
import os
from pathlib import Path
def resolve_project_root():

    # Used automatically by run_pipeline.py
    env_root = os.environ.get(
        "DATA_ENGINEERING_R62_ROOT"
    )

    if env_root:
        root = Path(env_root).resolve()

        if (
            (root / "data").exists()
            and
            (root / "notebooks").exists()
        ):
            return root


    # Interactive notebook execution
    cwd = Path.cwd().resolve()

    candidates = [
        cwd,
        cwd.parent
    ]

    for candidate in candidates:

        if (
            (candidate / "data").exists()
            and
            (candidate / "notebooks").exists()
        ):
            return candidate


    raise RuntimeError(
        "Could not locate DATA_ENGINEERING_R62 project root."
    )


PROJECT_ROOT = resolve_project_root()

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "\n✅ SUCCESS — Project root resolved."
)

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"

EMAIL_FILE = RAW / "email.csv"
FILE_FILE = RAW / "file.csv"
HTTP_FILE = RAW / "http.csv"
DEVICE_FILE = RAW / "device.csv"
LOGON_FILE = RAW / "logon.csv"
PSYCH_FILE = RAW / "psychometric.csv"

print("Raw path:", RAW)

for f in [
    EMAIL_FILE,
    FILE_FILE,
    HTTP_FILE,
    DEVICE_FILE,
    LOGON_FILE,
    PSYCH_FILE
]:
    print(f.name, "→", f.exists())

Project root: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62

✅ SUCCESS — Project root resolved.
Raw path: C:\Users\HP\Documents\FYP\DATA_ENGINEERING_R62\data\raw\r6.2
email.csv → True
file.csv → True
http.csv → True
device.csv → True
logon.csv → True
psychometric.csv → True


In [9]:
# =========================================================
# FILE SIZE SUMMARY
# =========================================================

files = [
    EMAIL_FILE,
    FILE_FILE,
    HTTP_FILE,
    DEVICE_FILE,
    LOGON_FILE,
    PSYCH_FILE
]

size_rows = []

for f in files:
    size_rows.append({
        "file": f.name,
        "size_GB": round(
            f.stat().st_size / (1024 ** 3),
            3
        )
    })

file_sizes = pd.DataFrame(size_rows)

print(
    file_sizes.to_string(index=False)
)

            file  size_GB
       email.csv    7.530
        file.csv    1.240
        http.csv   84.036
      device.csv    0.130
       logon.csv    0.225
psychometric.csv    0.000


In [11]:
# =========================================================
# READ SCHEMAS ONLY
# =========================================================

def show_schema(file_path, name):

    path = str(file_path).replace("\\", "/")

    schema = duckdb.sql(f"""
        DESCRIBE
        SELECT *
        FROM read_csv_auto(
            '{path}',
            all_varchar = TRUE,
            sample_size = 10000
        )
    """).df()

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print(
        schema[
            ["column_name", "column_type"]
        ].to_string(index=False)
    )


show_schema(
    EMAIL_FILE,
    "EMAIL SCHEMA"
)

show_schema(
    FILE_FILE,
    "FILE SCHEMA"
)

show_schema(
    HTTP_FILE,
    "HTTP SCHEMA"
)

show_schema(
    DEVICE_FILE,
    "DEVICE SCHEMA"
)

show_schema(
    LOGON_FILE,
    "LOGON SCHEMA"
)


EMAIL SCHEMA
column_name column_type
         id     VARCHAR
       date     VARCHAR
       user     VARCHAR
         pc     VARCHAR
         to     VARCHAR
         cc     VARCHAR
        bcc     VARCHAR
       from     VARCHAR
   activity     VARCHAR
       size     VARCHAR
attachments     VARCHAR
    content     VARCHAR

FILE SCHEMA
         column_name column_type
                  id     VARCHAR
                date     VARCHAR
                user     VARCHAR
                  pc     VARCHAR
            filename     VARCHAR
            activity     VARCHAR
  to_removable_media     VARCHAR
from_removable_media     VARCHAR
             content     VARCHAR

HTTP SCHEMA
column_name column_type
         id     VARCHAR
       date     VARCHAR
       user     VARCHAR
         pc     VARCHAR
        url     VARCHAR
   activity     VARCHAR
    content     VARCHAR

DEVICE SCHEMA
column_name column_type
         id     VARCHAR
       date     VARCHAR
       user     VARCHAR
         pc    

In [13]:
# =========================================================
# ACTIVITY VALUES
# =========================================================

def get_activity_values(file_path, name):

    path = str(file_path).replace("\\", "/")

    result = duckdb.sql(f"""
        SELECT
            activity,
            COUNT(*) AS events

        FROM read_csv_auto(
            '{path}',
            all_varchar = TRUE,
            sample_size = 10000
        )

        GROUP BY activity
        ORDER BY events DESC
    """).df()

    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    print(
        result.to_string(index=False)
    )


get_activity_values(
    EMAIL_FILE,
    "EMAIL ACTIVITY"
)

get_activity_values(
    FILE_FILE,
    "FILE ACTIVITY"
)

get_activity_values(
    HTTP_FILE,
    "HTTP ACTIVITY"
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


EMAIL ACTIVITY
activity  events
    View 7162491
    Send 3832466

FILE ACTIVITY
   activity  events
  File Copy  725458
  File Open  657500
File Delete  380799
 File Write  251126


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


HTTP ACTIVITY
    activity    events
   WWW Visit 115401178
WWW Download   1413637
  WWW Upload    210401


In [15]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"

EMAIL_FILE = RAW / "email.csv"
FILE_FILE = RAW / "file.csv"
HTTP_FILE = RAW / "http.csv"

email_path = str(EMAIL_FILE).replace("\\", "/")
file_path = str(FILE_FILE).replace("\\", "/")
http_path = str(HTTP_FILE).replace("\\", "/")

In [17]:
email_users = duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user) AS users_with_send,
        COUNT(*) AS send_events
    FROM read_csv_auto(
        '{email_path}',
        all_varchar = TRUE
    )
    WHERE activity = 'Send'
""").df()

print("=" * 100)
print("EMAIL SEND")
print("=" * 100)
print(email_users.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

EMAIL SEND
 users_with_send  send_events
            4000      3832466


In [19]:
external_email_users = duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user) AS users_with_external_email,
        COUNT(*) AS external_email_events
    FROM read_csv_auto(
        '{email_path}',
        all_varchar = TRUE
    )
    WHERE activity = 'Send'
      AND (
            LOWER(COALESCE("to", '')) NOT LIKE '%@dtaa.com%'
         OR LOWER(COALESCE(cc, '')) NOT LIKE '%@dtaa.com%'
         OR LOWER(COALESCE(bcc, '')) NOT LIKE '%@dtaa.com%'
      )
""").df()

print("\n" + "=" * 100)
print("EXTERNAL EMAIL")
print("=" * 100)
print(external_email_users.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


EXTERNAL EMAIL
 users_with_external_email  external_email_events
                      4000                3658598


In [ ]:
# =========================================================
# TRUE EXTERNAL EMAIL + TRUE 3-CHANNEL OVERLAP
# =========================================================

true_external_overlap = duckdb.sql(f"""
WITH sent_email AS (

    SELECT
        id,
        user,

        REPLACE(
            CONCAT_WS(
                ';',
                COALESCE("to", ''),
                COALESCE(cc, ''),
                COALESCE(bcc, '')
            ),
            ',',
            ';'
        ) AS recipients

    FROM read_csv_auto(
        '{email_path}',
        all_varchar = TRUE
    )

    WHERE activity = 'Send'
),

recipient_rows AS (

    SELECT
        s.id,
        s.user,
        TRIM(r.recipient) AS recipient

    FROM sent_email s,
    UNNEST(
        STRING_SPLIT(
            s.recipients,
            ';'
        )
    ) AS r(recipient)

    WHERE TRIM(r.recipient) <> ''
),

external_email_events AS (

    SELECT DISTINCT
        id,
        user

    FROM recipient_rows

    WHERE LOWER(recipient)
          NOT LIKE '%@dtaa.com'
),

external_email_users AS (

    SELECT DISTINCT user
    FROM external_email_events
),

usb_users AS (

    SELECT DISTINCT user

    FROM read_csv_auto(
        '{file_path}',
        all_varchar = TRUE
    )

    WHERE activity = 'File Copy'

      AND LOWER(
            COALESCE(
                to_removable_media,
                ''
            )
          ) IN (
              'true',
              '1',
              'yes'
          )
),

http_users AS (

    SELECT DISTINCT user

    FROM read_csv_auto(
        '{http_path}',
        all_varchar = TRUE
    )

    WHERE activity = 'WWW Upload'
)

SELECT

    (
        SELECT COUNT(DISTINCT user)
        FROM external_email_events
    ) AS users_with_true_external_email,

    (
        SELECT COUNT(*)
        FROM external_email_events
    ) AS true_external_email_events,

    (
        SELECT COUNT(*)
        FROM external_email_users e
        INNER JOIN usb_users u
            ON e.user = u.user
        INNER JOIN http_users h
            ON e.user = h.user
    ) AS users_with_true_all_3_channels
""").df()


print("=" * 100)
print("TRUE EXTERNAL EMAIL + CROSS-CHANNEL OVERLAP")
print("=" * 100)

print(
    true_external_overlap.to_string(
        index=False
    )
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [24]:
usb_users = duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user) AS users_with_usb_copy,
        COUNT(*) AS usb_copy_events
    FROM read_csv_auto(
        '{file_path}',
        all_varchar = TRUE
    )
    WHERE activity = 'File Copy'
      AND LOWER(COALESCE(to_removable_media, '')) IN (
            'true', '1', 'yes'
      )
""").df()

print("\n" + "=" * 100)
print("USB COPY TO REMOVABLE MEDIA")
print("=" * 100)
print(usb_users.to_string(index=False))


USB COPY TO REMOVABLE MEDIA
 users_with_usb_copy  usb_copy_events
                 786           333611


In [26]:
http_upload_users = duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT user) AS users_with_www_upload,
        COUNT(*) AS www_upload_events
    FROM read_csv_auto(
        '{http_path}',
        all_varchar = TRUE
    )
    WHERE activity = 'WWW Upload'
""").df()

print("\n" + "=" * 100)
print("HTTP WWW UPLOAD")
print("=" * 100)
print(http_upload_users.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


HTTP WWW UPLOAD
 users_with_www_upload  www_upload_events
                   619             210401


In [28]:
cross_channel_users = duckdb.sql(f"""
    WITH email_users AS (
        SELECT DISTINCT user
        FROM read_csv_auto(
            '{email_path}',
            all_varchar = TRUE
        )
        WHERE activity = 'Send'
    ),

    usb_users AS (
        SELECT DISTINCT user
        FROM read_csv_auto(
            '{file_path}',
            all_varchar = TRUE
        )
        WHERE activity = 'File Copy'
          AND LOWER(COALESCE(to_removable_media, ''))
              IN ('true', '1', 'yes')
    ),

    http_users AS (
        SELECT DISTINCT user
        FROM read_csv_auto(
            '{http_path}',
            all_varchar = TRUE
        )
        WHERE activity = 'WWW Upload'
    )

    SELECT
        COUNT(*) AS users_with_all_3_channels

    FROM email_users e
    INNER JOIN usb_users u
        ON e.user = u.user
    INNER JOIN http_users h
        ON e.user = h.user
""").df()

print("\n" + "=" * 100)
print("CROSS-CHANNEL OVERLAP")
print("=" * 100)
print(cross_channel_users.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


CROSS-CHANNEL OVERLAP
 users_with_all_3_channels
                       322


In [72]:
from pathlib import Path
import pandas as pd
ANSWERS = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "r6.2"
    / "answers"
)

INSIDERS_FILE = ANSWERS / "insiders.csv"

insiders = pd.read_csv(
    INSIDERS_FILE,
    dtype=str
)

print("Columns:")
print(insiders.columns.tolist())

print("\nShape:")
print(insiders.shape)

print("\nSample:")
print(
    insiders.head(10).to_string(index=False)
)

Columns:
['dataset', 'scenario', 'details', 'user', 'start', 'end']

Shape:
(191, 6)

Sample:
dataset scenario            details    user               start                 end
      2        1             r2.csv ONS0995    3/6/2010 1:41:56   3/20/2010 8:10:12
    3.1        1         r3.1-1.csv CSF0929 07/01/2010 01:24:58 07/16/2010 06:52:00
    3.1        2         r3.1-2.csv CCH0959 08/02/2010 10:34:31 09/30/2010 15:04:03
    3.2        1         r3.2-1.csv RCW0822 09/29/2010 21:10:27 10/15/2010 06:34:52
    3.2        2         r3.2-2.csv JCE0258 07/12/2010 08:16:02 09/03/2010 16:16:29
    4.1        1         r4.1-1.csv ABB0427 08/21/2010 05:20:34 08/27/2010 05:39:51
    4.1        2         r4.1-2.csv HFC0492 08/23/2010 10:40:56 10/14/2010 17:11:42
    4.1        3         r4.1-3.csv KTW0365 07/08/2010 11:51:00 07/09/2010 20:54:16
    4.2        1 r4.2-1-AAM0658.csv AAM0658 10/23/2010 01:34:19 10/29/2010 05:23:28
    4.2        1 r4.2-1-AJR0932.csv AJR0932 09/10/2010 19:12:01 09

In [32]:
# =========================================================
# FILTER R6.2 ONLY
# =========================================================

insiders["dataset_clean"] = (
    insiders["dataset"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace("r", "", regex=False)
)

r62_gt = insiders[
    insiders["dataset_clean"] == "6.2"
].copy()

print("=" * 100)
print("R6.2 OFFICIAL GROUND TRUTH")
print("=" * 100)

print(
    r62_gt.to_string(index=False)
)

print("\nRows:", len(r62_gt))

print(
    "Unique malicious users:",
    r62_gt["user"].nunique()
)

print(
    "Scenarios:",
    sorted(
        r62_gt["scenario"].dropna().unique()
    )
)

R6.2 OFFICIAL GROUND TRUTH
dataset scenario    details    user               start                 end dataset_clean
    6.2        1 r6.2-1.csv ACM2278  8/18/2010 21:47:42 08/24/2010 03:48:51           6.2
    6.2        2 r6.2-2.csv CMP2946 02/07/2011 12:28:06 03/04/2011 12:30:25           6.2
    6.2        3 r6.2-3.csv PLJ1771  8/12/2010 08:09:00 08/12/2010 16:11:39           6.2
    6.2        4 r6.2-4.csv CDE1846   /21/2011 11:43:39 04/25/2011 17:55:00           6.2
    6.2        5 r6.2-5.csv MBG3183 10/12/2010 13:21:59 10/12/2010 13:22:56           6.2

Rows: 5
Unique malicious users: 5
Scenarios: ['1', '2', '3', '4', '5']


In [34]:
print(
    r62_gt[
        [
            "scenario",
            "user",
            "start",
            "end",
            "details"
        ]
    ].to_string(index=False)
)

scenario    user               start                 end    details
       1 ACM2278  8/18/2010 21:47:42 08/24/2010 03:48:51 r6.2-1.csv
       2 CMP2946 02/07/2011 12:28:06 03/04/2011 12:30:25 r6.2-2.csv
       3 PLJ1771  8/12/2010 08:09:00 08/12/2010 16:11:39 r6.2-3.csv
       4 CDE1846   /21/2011 11:43:39 04/25/2011 17:55:00 r6.2-4.csv
       5 MBG3183 10/12/2010 13:21:59 10/12/2010 13:22:56 r6.2-5.csv


In [36]:
print("=" * 100)
print("R6.2 OFFICIAL GROUND TRUTH")
print("=" * 100)

print(
    r62_gt[
        ["dataset", "scenario", "details", "user", "start", "end"]
    ].to_string(index=False)
)

print("\nRows:", len(r62_gt))
print("Unique malicious users:", r62_gt["user"].nunique())

R6.2 OFFICIAL GROUND TRUTH
dataset scenario    details    user               start                 end
    6.2        1 r6.2-1.csv ACM2278  8/18/2010 21:47:42 08/24/2010 03:48:51
    6.2        2 r6.2-2.csv CMP2946 02/07/2011 12:28:06 03/04/2011 12:30:25
    6.2        3 r6.2-3.csv PLJ1771  8/12/2010 08:09:00 08/12/2010 16:11:39
    6.2        4 r6.2-4.csv CDE1846   /21/2011 11:43:39 04/25/2011 17:55:00
    6.2        5 r6.2-5.csv MBG3183 10/12/2010 13:21:59 10/12/2010 13:22:56

Rows: 5
Unique malicious users: 5


In [38]:
import csv

scenario_files = [
    ANSWERS / f"r6.2-{i}.csv"
    for i in range(1, 6)
]

for f in scenario_files:

    print("\n" + "=" * 100)
    print(f.name)
    print("=" * 100)

    with open(
        f,
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as file:

        reader = csv.reader(file)

        for row_num, row in enumerate(reader, start=1):

            print(
                f"Row {row_num:02d} | "
                f"fields={len(row)} | "
                f"{row}"
            )

            if row_num >= 8:
                break


r6.2-1.csv
Row 01 | fields=6 | ['logon', '{G3R8-G0BG91SZ-1111ZKPB}', '08/18/2010 21:47:42', 'ACM2278', 'PC-8431', 'Logon']
Row 02 | fields=7 | ['device', '{F0N3-C5GS70QE-4575NRWF}', '08/18/2010 22:59:20', 'ACM2278', 'PC-8431', 'R:\\;R:\\52G6677;R:\\782bxm8;R:\\ACM2278', 'Connect']
Row 03 | fields=11 | ['file', '{L8U4-E1KC17HI-2295QTAV}', '08/19/2010 01:34:19', 'ACM2278', 'PC-8431', 'R:\\52G6677\\B7RGYJZC.jpg', 'File Open', 'False', 'True', 'The_Real_Story_About_DTAA', 'FF-D8 ']
Row 04 | fields=11 | ['file', '{Z8S0-I9WK82HT-2702OXHW}', '08/19/2010 01:37:20', 'ACM2278', 'PC-8431', 'R:\\52G6677\\B7RGYJZC.jpg', 'File Open', 'False', 'True', 'The_Real_Story_About_DTAA', 'FF-D8 ']
Row 05 | fields=11 | ['file', '{M3U7-F9ZW51CX-5408MNTW}', '08/19/2010 01:38:10', 'ACM2278', 'PC-8431', 'R:\\52G6677\\7QO6RIRR.txt', 'File Open', 'False', 'True', 'The_Real_Story_About_DTAA', '33-4E-42-56 Samantabhadra (2nd century CE) and Siddhasena Divakara (3rd century CE) further fine-tuned Jain epistemology an

In [40]:
def read_ragged_csv(path):

    rows = []

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace",
        newline=""
    ) as file:

        reader = csv.reader(file)
        rows = list(reader)

    max_cols = max(
        len(row)
        for row in rows
    )

    padded = [
        row + [None] * (max_cols - len(row))
        for row in rows
    ]

    columns = [
        f"col_{i+1}"
        for i in range(max_cols)
    ]

    return pd.DataFrame(
        padded,
        columns=columns
    )


for f in scenario_files:

    df = read_ragged_csv(f)

    print("\n" + "=" * 100)
    print(f.name, df.shape)
    print("=" * 100)

    print(
        df.head(10).to_string(index=False)
    )


r6.2-1.csv (22, 11)
 col_1                    col_2               col_3   col_4   col_5                                col_6      col_7 col_8 col_9                    col_10                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [42]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PATHS
# =========================================================

RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"

EMAIL_FILE = RAW / "email.csv"
FILE_FILE = RAW / "file.csv"

email_path = str(EMAIL_FILE).replace("\\", "/")
file_path = str(FILE_FILE).replace("\\", "/")

print("EMAIL:", EMAIL_FILE.exists())
print("FILE :", FILE_FILE.exists())

EMAIL: True
FILE : True


In [44]:
scenario5_files = [
    "46BUD2JL.zip",
    "WO83A8RQ.jpg",
    "756O1A4U.txt"
]

names_sql = ", ".join(
    f"'{name.lower()}'"
    for name in scenario5_files
)

# =========================================================
# 1. SEARCH IN FILE.CSV
# =========================================================

file_matches = duckdb.sql(f"""
    SELECT
        id,
        date,
        user,
        pc,
        filename,
        activity,
        to_removable_media,
        from_removable_media
    FROM read_csv_auto(
        '{file_path}',
        all_varchar = TRUE
    )
    WHERE LOWER(filename) IN ({names_sql})
    ORDER BY filename, date
""").df()

print("=" * 100)
print("SCENARIO 5 FILENAMES IN FILE.CSV")
print("=" * 100)

print("Matches:", len(file_matches))

print(
    file_matches.to_string(index=False)
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SCENARIO 5 FILENAMES IN FILE.CSV
Matches: 0
Empty DataFrame
Columns: [id, date, user, pc, filename, activity, to_removable_media, from_removable_media]
Index: []


In [46]:
from pathlib import Path
import duckdb
import pandas as pd

# =========================================================
# PROJECT PATHS
# =========================================================


RAW = PROJECT_ROOT / "data" / "raw" / "r6.2"

EMAIL_FILE = RAW / "email.csv"
FILE_FILE = RAW / "file.csv"
HTTP_FILE = RAW / "http.csv"

email_path = str(EMAIL_FILE).replace("\\", "/")
file_path = str(FILE_FILE).replace("\\", "/")
http_path = str(HTTP_FILE).replace("\\", "/")


# =========================================================
# OFFICIAL R6.2 MALICIOUS USERS
# =========================================================

malicious_users = {
    "ACM2278",
    "CMP2946",
    "PLJ1771",
    "CDE1846",
    "MBG3183"
}

malicious_sql = ", ".join(
    f"'{user}'"
    for user in malicious_users
)


# =========================================================
# CHECK PATHS
# =========================================================

print("Email exists:", EMAIL_FILE.exists())
print("File exists :", FILE_FILE.exists())
print("HTTP exists :", HTTP_FILE.exists())

print("\nMalicious users excluded:")
print(sorted(malicious_users))

assert EMAIL_FILE.exists()
assert FILE_FILE.exists()
assert HTTP_FILE.exists()

print("\n✅ SUCCESS — Paths and malicious users are ready.")

Email exists: True
File exists : True
HTTP exists : True

Malicious users excluded:
['ACM2278', 'CDE1846', 'CMP2946', 'MBG3183', 'PLJ1771']

✅ SUCCESS — Paths and malicious users are ready.


In [48]:
# =========================================================
# BENIGN EXTERNAL EMAIL ACTIVITY
# =========================================================

email_daily = duckdb.sql(f"""
WITH sent_email AS (

    SELECT
        id,
        user,

        CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        ) AS day,

        REPLACE(
            CONCAT_WS(
                ';',
                COALESCE("to", ''),
                COALESCE(cc, ''),
                COALESCE(bcc, '')
            ),
            ',',
            ';'
        ) AS recipients

    FROM read_csv_auto(
        '{email_path}',
        all_varchar = TRUE
    )

    WHERE activity = 'Send'
      AND user NOT IN ({malicious_sql})
),

recipient_rows AS (

    SELECT
        id,
        user,
        day,
        TRIM(recipient) AS recipient

    FROM sent_email,
    UNNEST(
        STRING_SPLIT(
            recipients,
            ';'
        )
    ) AS t(recipient)

    WHERE TRIM(recipient) <> ''
),

external_events AS (

    SELECT DISTINCT
        id,
        user,
        day

    FROM recipient_rows

    WHERE LOWER(recipient)
          NOT LIKE '%@dtaa.com'
)

SELECT
    user,

    COUNT(
        DISTINCT day
    ) AS external_email_days,

    COUNT(*) AS external_email_events

FROM external_events

GROUP BY user

ORDER BY user
""").df()


print("Users with benign external email activity:")
print(email_daily["user"].nunique())

print("\nSample:")
print(
    email_daily.head().to_string(index=False)
)

assert len(email_daily) > 0

print("\n✅ SUCCESS — External Email activity calculated.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Users with benign external email activity:
3994

Sample:
   user  external_email_days  external_email_events
AAB0162                  218                    390
AAB0398                  282                    696
AAC0610                  131                    156
AAC0668                  248                    504
AAC3270                   27                     27

✅ SUCCESS — External Email activity calculated.


In [50]:
# =========================================================
# BENIGN USB COPY ACTIVITY
# =========================================================

usb_daily = duckdb.sql(f"""
SELECT

    user,

    COUNT(
        DISTINCT CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        )
    ) AS usb_copy_days,

    COUNT(*) AS usb_copy_events

FROM read_csv_auto(
    '{file_path}',
    all_varchar = TRUE
)

WHERE activity = 'File Copy'

  AND LOWER(
        COALESCE(
            to_removable_media,
            ''
        )
      ) IN (
        'true',
        '1',
        'yes'
      )

  AND user NOT IN ({malicious_sql})

GROUP BY user

ORDER BY user
""").df()


print("Users with benign USB copy activity:")
print(usb_daily["user"].nunique())

print("\nSample:")
print(
    usb_daily.head().to_string(index=False)
)

assert len(usb_daily) > 0

print("\n✅ SUCCESS — USB Copy activity calculated.")

Users with benign USB copy activity:
783

Sample:
   user  usb_copy_days  usb_copy_events
AAC0610            186              369
AAF0819            292              646
AAP0352            104              140
AAP1919             65               90
ABD3426            463             2590

✅ SUCCESS — USB Copy activity calculated.


In [52]:
# =========================================================
# BENIGN HTTP WWW UPLOAD ACTIVITY
# =========================================================

http_daily = duckdb.sql(f"""
SELECT

    user,

    COUNT(
        DISTINCT CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        )
    ) AS http_upload_days,

    COUNT(*) AS http_upload_events

FROM read_csv_auto(
    '{http_path}',
    all_varchar = TRUE
)

WHERE activity = 'WWW Upload'

  AND user NOT IN ({malicious_sql})

GROUP BY user

ORDER BY user
""").df()


print("Users with benign WWW Upload activity:")
print(http_daily["user"].nunique())

print("\nSample:")
print(
    http_daily.head().to_string(index=False)
)

assert len(http_daily) > 0

print("\n✅ SUCCESS — HTTP WWW Upload activity calculated.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Users with benign WWW Upload activity:
615

Sample:
   user  http_upload_days  http_upload_events
AAS2987                84                 150
ABD3426               102                 151
ABH0349                98                 138
ABM3203                99                 151
ABO1173                77                 116

✅ SUCCESS — HTTP WWW Upload activity calculated.


In [54]:
# =========================================================
# THREE-CHANNEL BENIGN USERS
# =========================================================

channel_frequency = (

    email_daily

    .merge(
        usb_daily,
        on="user",
        how="inner"
    )

    .merge(
        http_daily,
        on="user",
        how="inner"
    )

)


print("=" * 100)
print("BENIGN THREE-CHANNEL USERS")
print("=" * 100)

print(
    "Total users:",
    channel_frequency["user"].nunique()
)

print("\nSample:")
print(
    channel_frequency.head(10)
    .to_string(index=False)
)

assert len(channel_frequency) > 0

print(
    "\n✅ SUCCESS — Three-channel benign users identified."
)

BENIGN THREE-CHANNEL USERS
Total users: 319

Sample:
   user  external_email_days  external_email_events  usb_copy_days  usb_copy_events  http_upload_days  http_upload_events
ABD3426                  403                   1278            463             2590               102                 151
ABH0349                   49                     49             90              300                98                 138
ABO1173                  222                    422             81              111                77                 116
ACB0701                  140                    168             18               21               119                 394
ACG3819                  110                    129            116              170                94                 151
ACV1946                  234                    450            105              174                71                 104
ADB1105                  133                    234             81              126          

In [56]:
# =========================================================
# ACTIVE-DAY DISTRIBUTIONS
# =========================================================

print("=" * 100)
print("ACTIVE-DAY DISTRIBUTIONS")
print("=" * 100)


columns_to_check = [
    "external_email_days",
    "usb_copy_days",
    "http_upload_days"
]


for col in columns_to_check:

    print("\n" + "-" * 70)
    print(col.upper())
    print("-" * 70)

    print(
        channel_frequency[col]
        .describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
                0.95
            ]
        )
    )


print(
    "\n✅ SUCCESS — Active-day distributions calculated."
)

ACTIVE-DAY DISTRIBUTIONS

----------------------------------------------------------------------
EXTERNAL_EMAIL_DAYS
----------------------------------------------------------------------
count    319.000000
mean     201.655172
std       95.937514
min       27.000000
10%       51.000000
25%      122.000000
50%      232.000000
75%      271.500000
90%      304.200000
95%      329.200000
max      437.000000
Name: external_email_days, dtype: float64

----------------------------------------------------------------------
USB_COPY_DAYS
----------------------------------------------------------------------
count    319.000000
mean     112.949843
std      106.714973
min        4.000000
10%       16.000000
25%       34.000000
50%       81.000000
75%      145.500000
90%      280.200000
95%      355.000000
max      486.000000
Name: usb_copy_days, dtype: float64

----------------------------------------------------------------------
HTTP_UPLOAD_DAYS
------------------------------------------------

In [58]:
# =========================================================
# MINIMUM CHANNEL EXPERIENCE
# =========================================================

channel_frequency[
    "min_channel_active_days"
] = channel_frequency[
    [
        "external_email_days",
        "usb_copy_days",
        "http_upload_days"
    ]
].min(axis=1)


print("=" * 100)
print("MINIMUM CHANNEL ACTIVE DAYS")
print("=" * 100)

print(
    channel_frequency[
        "min_channel_active_days"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)


print(
    "\n✅ SUCCESS — Minimum channel experience calculated."
)

MINIMUM CHANNEL ACTIVE DAYS
count    319.000000
mean      60.793103
std       34.662358
min        4.000000
10%       16.000000
25%       33.000000
50%       58.000000
75%       84.000000
90%      108.600000
95%      121.000000
max      162.000000
Name: min_channel_active_days, dtype: float64

✅ SUCCESS — Minimum channel experience calculated.


In [60]:
# =========================================================
# CANDIDATE COUNTS AT DIFFERENT THRESHOLDS
# =========================================================

thresholds = [
    1,
    5,
    10,
    20,
    30,
    50
]

rows = []

for threshold in thresholds:

    users = (
        channel_frequency[
            "min_channel_active_days"
        ] >= threshold
    ).sum()

    rows.append({
        "minimum_days_per_channel":
            threshold,

        "eligible_users":
            users
    })


threshold_summary = pd.DataFrame(
    rows
)


print("=" * 100)
print("THREE-CHANNEL USER AVAILABILITY")
print("=" * 100)

print(
    threshold_summary.to_string(
        index=False
    )
)


print(
    "\n✅ SUCCESS — Candidate threshold summary calculated."
)

THREE-CHANNEL USER AVAILABILITY
 minimum_days_per_channel  eligible_users
                        1             319
                        5             318
                       10             305
                       20             274
                       30             246
                       50             189

✅ SUCCESS — Candidate threshold summary calculated.


In [62]:
# =========================================================
# CHANNEL TEMPORAL COVERAGE
# =========================================================

email_temporal = duckdb.sql(f"""
WITH sent_email AS (

    SELECT
        id,
        user,

        CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        ) AS day,

        REPLACE(
            CONCAT_WS(
                ';',
                COALESCE("to", ''),
                COALESCE(cc, ''),
                COALESCE(bcc, '')
            ),
            ',',
            ';'
        ) AS recipients

    FROM read_csv_auto(
        '{email_path}',
        all_varchar = TRUE
    )

    WHERE activity = 'Send'
      AND user NOT IN ({malicious_sql})
),

recipient_rows AS (

    SELECT
        id,
        user,
        day,
        TRIM(recipient) AS recipient

    FROM sent_email,
    UNNEST(
        STRING_SPLIT(
            recipients,
            ';'
        )
    ) AS t(recipient)

    WHERE TRIM(recipient) <> ''
),

external_events AS (

    SELECT DISTINCT
        id,
        user,
        day

    FROM recipient_rows

    WHERE LOWER(recipient)
          NOT LIKE '%@dtaa.com'
)

SELECT
    user,
    MIN(day) AS email_first_day,
    MAX(day) AS email_last_day

FROM external_events

GROUP BY user
""").df()


usb_temporal = duckdb.sql(f"""
SELECT
    user,

    MIN(
        CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        )
    ) AS usb_first_day,

    MAX(
        CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        )
    ) AS usb_last_day

FROM read_csv_auto(
    '{file_path}',
    all_varchar = TRUE
)

WHERE activity = 'File Copy'

  AND LOWER(
        COALESCE(
            to_removable_media,
            ''
        )
      ) IN ('true', '1', 'yes')

  AND user NOT IN ({malicious_sql})

GROUP BY user
""").df()


http_temporal = duckdb.sql(f"""
SELECT
    user,

    MIN(
        CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        )
    ) AS http_first_day,

    MAX(
        CAST(
            strptime(
                date,
                '%m/%d/%Y %H:%M:%S'
            )
            AS DATE
        )
    ) AS http_last_day

FROM read_csv_auto(
    '{http_path}',
    all_varchar = TRUE
)

WHERE activity = 'WWW Upload'

  AND user NOT IN ({malicious_sql})

GROUP BY user
""").df()


print("✅ SUCCESS — Temporal coverage calculated.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ SUCCESS — Temporal coverage calculated.


In [64]:
# =========================================================
# COMMON THREE-CHANNEL TIME WINDOW
# =========================================================

temporal_overlap = (

    channel_frequency

    .merge(
        email_temporal,
        on="user",
        how="left"
    )

    .merge(
        usb_temporal,
        on="user",
        how="left"
    )

    .merge(
        http_temporal,
        on="user",
        how="left"
    )
)


# Latest starting date among the three channels
temporal_overlap[
    "common_start"
] = temporal_overlap[
    [
        "email_first_day",
        "usb_first_day",
        "http_first_day"
    ]
].max(axis=1)


# Earliest ending date among the three channels
temporal_overlap[
    "common_end"
] = temporal_overlap[
    [
        "email_last_day",
        "usb_last_day",
        "http_last_day"
    ]
].min(axis=1)


temporal_overlap[
    "common_period_days"
] = (
    temporal_overlap["common_end"]
    - temporal_overlap["common_start"]
).dt.days + 1


print("=" * 100)
print("COMMON THREE-CHANNEL PERIOD")
print("=" * 100)

print(
    temporal_overlap[
        "common_period_days"
    ].describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95
        ]
    )
)

print("\n✅ SUCCESS — Common three-channel period calculated.")

COMMON THREE-CHANNEL PERIOD
count    319.000000
mean     463.407524
std       90.051820
min       22.000000
10%      389.400000
25%      475.500000
50%      498.000000
75%      505.000000
90%      508.000000
95%      509.000000
max      515.000000
Name: common_period_days, dtype: float64

✅ SUCCESS — Common three-channel period calculated.


In [66]:
# =========================================================
# FINAL FEASIBILITY CANDIDATE SUMMARY
# =========================================================

day_thresholds = [
    10,
    20,
    30,
    50
]

common_window_thresholds = [
    30,
    60,
    90,
    180
]

rows = []

for activity_days in day_thresholds:

    for common_days in common_window_thresholds:

        eligible = temporal_overlap[
            (
                temporal_overlap[
                    "min_channel_active_days"
                ] >= activity_days
            )
            &
            (
                temporal_overlap[
                    "common_period_days"
                ] >= common_days
            )
        ]

        rows.append({

            "min_active_days_each_channel":
                activity_days,

            "min_common_period_days":
                common_days,

            "eligible_users":
                len(eligible)
        })


final_feasibility = pd.DataFrame(rows)

print("=" * 100)
print("FINAL FEASIBILITY MATRIX")
print("=" * 100)

print(
    final_feasibility.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Final feasibility matrix calculated."
)

FINAL FEASIBILITY MATRIX
 min_active_days_each_channel  min_common_period_days  eligible_users
                           10                      30             305
                           10                      60             304
                           10                      90             301
                           10                     180             297
                           20                      30             274
                           20                      60             274
                           20                      90             273
                           20                     180             272
                           30                      30             246
                           30                      60             246
                           30                      90             246
                           30                     180             246
                           50                      30            

In [68]:
# =========================================================
# FINAL FEASIBILITY CANDIDATE SUMMARY
# =========================================================

day_thresholds = [
    10,
    20,
    30,
    50
]

common_window_thresholds = [
    30,
    60,
    90,
    180
]

rows = []

for activity_days in day_thresholds:

    for common_days in common_window_thresholds:

        eligible = temporal_overlap[
            (
                temporal_overlap[
                    "min_channel_active_days"
                ] >= activity_days
            )
            &
            (
                temporal_overlap[
                    "common_period_days"
                ] >= common_days
            )
        ]

        rows.append({

            "min_active_days_each_channel":
                activity_days,

            "min_common_period_days":
                common_days,

            "eligible_users":
                len(eligible)
        })


final_feasibility = pd.DataFrame(rows)

print("=" * 100)
print("FINAL FEASIBILITY MATRIX")
print("=" * 100)

print(
    final_feasibility.to_string(
        index=False
    )
)

print(
    "\n✅ SUCCESS — Final feasibility matrix calculated."
)

FINAL FEASIBILITY MATRIX
 min_active_days_each_channel  min_common_period_days  eligible_users
                           10                      30             305
                           10                      60             304
                           10                      90             301
                           10                     180             297
                           20                      30             274
                           20                      60             274
                           20                      90             273
                           20                     180             272
                           30                      30             246
                           30                      60             246
                           30                      90             246
                           30                     180             246
                           50                      30            

In [70]:
RESULTS = PROJECT_ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

channel_frequency.to_csv(
    RESULTS / "r62_channel_frequency_audit.csv",
    index=False
)

temporal_overlap.to_csv(
    RESULTS / "r62_temporal_overlap_audit.csv",
    index=False
)

final_feasibility.to_csv(
    RESULTS / "r62_final_feasibility_matrix.csv",
    index=False
)

primary_candidates = temporal_overlap[
    (temporal_overlap["min_channel_active_days"] >= 50)
    &
    (temporal_overlap["common_period_days"] >= 180)
].copy()

primary_candidates.to_csv(
    RESULTS / "r62_primary_candidate_users.csv",
    index=False
)

print("Primary candidates:", len(primary_candidates))

print("\nSaved files:")
print("- r62_channel_frequency_audit.csv")
print("- r62_temporal_overlap_audit.csv")
print("- r62_final_feasibility_matrix.csv")
print("- r62_primary_candidate_users.csv")

assert len(primary_candidates) == 189

print("\n✅ SUCCESS — Feasibility audit frozen and saved.")

Primary candidates: 189

Saved files:
- r62_channel_frequency_audit.csv
- r62_temporal_overlap_audit.csv
- r62_final_feasibility_matrix.csv
- r62_primary_candidate_users.csv

✅ SUCCESS — Feasibility audit frozen and saved.
